# Homework 1: Contextual Spell Checking

## 1. Task Summary

### Objective
The goal is to develop an improved spell checker that corrects typos in sentences using a masked language model (DistilBERT). Given a sentence with one or more misspelled words, our system should predict the correct word.

### Problem Setup
- **Input**: Sentences with marked typo positions
- **Output**: Corrected sentences with the same format
- **Baseline**: Uses transformer's top-1 prediction from 20 candidates
- **Challenge**: Select the most appropriate correction from transformer predictions

## 2. Method Description

### 2.1 Baseline Approach
The baseline implementation (`default.py`) uses a simple strategy:
1. Masks the typo word in the sentence
2. Uses DistilBERT to generate **top 20** candidate replacements
3. Selects the **highest-scoring candidate** (by transformer probability)

**Limitations:**
- Only considers 20 candidates (may miss the correct word)
- Ignores edit distance between typo and candidates
- Doesn't account for how similar the candidate looks to the typo

### 2.2 Our Improved Approach

We developed a multi-component selection strategy with the following improvements:

#### **Improvement 1: Increased Candidate Pool**
- Increased from **20 to 200** candidate predictions
- Analysis showed only **58.41%** of ground truth words are in top-20 candidates generated by transformer
- With 200 candidates, **76.04%** of ground truths are now available

#### **Improvement 2: Damerau-Levenshtein Edit Distance & String Similarity Score**
- Implemented custom edit distance calculation
  - Handles **4 operations**: insertion, deletion, substitution, and **transposition**
  - More linguistically appropriate for spelling errors
- Uses `SequenceMatcher` from Python's `difflib`
  - Provides ratio of matching characters between typo and candidate
  - Captures visual/string similarity beyond just edit operations

#### **Improvement 3: Weighted Combination Selection**
Our final selection method uses a weighted combination of three normalized scores:

1. **Edit Distance Score** (weight: 0.45)
   ```python
   edit_distance_score = 1 - (distance / 7)  # max_distance = 7 from data
   ```
   - Linear normalization: closer matches get higher scores
   - Range: [0, 1], where 1 means distance=0 (but we filter those out)

2. **Similarity Score** (weight: 0.45)
   ```python
   similarity_score = SequenceMatcher(None, typo, candidate).ratio()
   ```
   - Already in [0, 1] range
   - Captures character-level similarity

3. **Transformer Score** (weight: 0.1)
   ```python
   transformer_score = model_probability  # from DistilBERT
   ```
   - Already in [0, 1] range
   - Represents contextual appropriateness

**Final Selection:**
```python
combined_score = (0.45 × edit_distance_score) + 
                 (0.45 × similarity_score) + 
                 (0.1 × transformer_score)
```
Select candidate with highest combined score (must have distance > 0)

**Weight Rationale:**
- Edit distance & Similarity (0.45 each): Primary signals for spelling correction
- Transformer score (0.1): Lower weight because it can sometimes prefer common words over the correct spelling
- Total = 1.0 for proper normalization

#### **Improvement 4: Capitalization Preservation**
- Checks if original typo starts with uppercase
- Capitalizes the selected correction accordingly

### 2.3 Comprehensive Analysis
To understand our method's performance and iterate on improvements, we implemented a detailed analysis mode that generates diagnostic files:

#### Analysis Files Generated:

**1. `dev_score_details.csv`** - Quantitative metrics for every correction
   - Contains side-by-side comparison of ground truth vs selected word
   - For each typo, records:
     - Typo word
     - Ground truth word and its metrics (transformer score, rank, edit distance, edit score, similarity)
     - Selected word and its metrics (same metrics as ground truth)
   - Enables systematic analysis of where and why our method fails

**2. `dev_error_details.txt`** - Qualitative analysis of incorrect predictions
   - Only logs cases where our prediction was wrong
   - For each error, shows:
     - Original sentence with typo marked
     - All top-200 predictions with their scores
     - Marks where ground truth and selected word appear in ranking
   - Helps identify patterns in failures (e.g., context-dependent errors)

#### How Analysis Mode Helped:

1. **Discovered the 24% limitation**: Found that ~24% of ground truths aren't in top-200, setting realistic expectations
2. **Guided weight selection**: CSV metrics showed which signals (edit distance, similarity, transformer) were most predictive
3. **Identified failure patterns**: Error logs revealed cases where context matters more than string similarity
4. **Validated improvements**: Quantitative comparison between different methods (distance-first vs weighted)

## 3. Results

### 3.1 Overall Performance

We incrementally added improvements and evaluated each configuration on the development set:
Note that
* Unavoidable Errors: Ground truth not in top-k candidates (~24% of cases when k=200, limitation of transformer results)
* Selection Errors: Ground truth available but wrong choice made (evaluate the selection method)

| Method | Accuracy | Total Errors | Unavoidable Errors * | Selection Errors * |
|--------|----------|--------------|----------------------|--------------------|
| **Baseline** (top-1 from 20 candidates) | 0.22 | 1348 | 717 | 631 |
| **Improvement 1** (transformer 200 candidates) | 0.22 | 1348 | 413 | 935 |
| **Improvement 2** (select candidate with smallest edit distance) | 0.66 | 593 | 413 | 180 |
| **Improvement 3** (use weighted combination to select word) | 0.66 | 581 | 413 | 168 |
| **Improvement 4** (preserve capitalization)| 0.74 | 455 | 413 | 42 |

#### Analysis of Results:

**Improvement 1 (200 candidates)**: No accuracy change (0.22 → 0.22), but reduced unavoidable errors by 42% (717 → 413), making 304 more correct words accessible. Selection errors increased (631 → 935) because the naive top-1 approach doesn't scale to larger candidate pools.

**Improvement 2 (Edit distance)**: **Breakthrough improvement** from 22% → 66% accuracy (+200%). Edit distance-based selection reduced selection errors by 81% (935 → 180), demonstrating it's the most powerful signal for spelling correction.

**Improvement 3 (Weighted combination)**: Modest refinement reducing selection errors by 7% (180 → 168). Combining edit distance, similarity, and transformer scores yields 12 fewer errors than edit distance alone.

**Improvement 4 (Capitalization)**: Improved from 66% → 74% (+12%), revealing 126 cases had correct words but wrong capitalization. After adding preservation logic, only 42 selection errors remain.

### 3.2 Key Findings from Analysis

Using our analysis mode output files, we discovered several important patterns:

#### Finding 1: Edit Distance Distribution

Most typos are within 1-4 edits from the correct word, validating our choice to prioritize edit distance.

In [1]:
import pandas as pd
import numpy as np

# Read the scores.csv file
df = pd.read_csv('../output/dev_score_details.csv')

# Distribution of edit distances
print(f"\nEdit Distance Distribution for Ground Truth Words:")
edit_dist_counts = df['ground_truth_edit_distance'].value_counts().sort_index()
for dist, count in edit_dist_counts.items():
    print(f"  Distance {dist}: {count} occurrences")


Edit Distance Distribution for Ground Truth Words:
  Distance 1: 1200 occurrences
  Distance 2: 403 occurrences
  Distance 3: 88 occurrences
  Distance 4: 24 occurrences
  Distance 5: 7 occurrences
  Distance 6: 1 occurrences
  Distance 7: 1 occurrences


#### Finding 2: Upper Bound on Accuracy (~76%)

**Critical limitation**: Approximately 24% of ground truth words are not in the top-200 transformer predictions. This sets a theoretical maximum accuracy of ~76%, regardless of how clever our selection method is.

In [2]:
import pandas as pd
import numpy as np

# Read the scores.csv file
df = pd.read_csv('../output/dev_score_details.csv')

num_na = df['ground_truth_transformer_score'].isna().sum()
total_entries = len(df)
na_percentage = (num_na / total_entries) * 100
print(f"\n\nTRANSFORMER SCORE N/A COUNT: {num_na} out of {total_entries} ({na_percentage:.2f}%)")

# How many ground truths were found in top 10, top 20, etc.
print(f"\nGround Truth Rank Distribution:")
if df['ground_truth_transformer_rank'].notna().any():
    in_top_5 = (df['ground_truth_transformer_rank'] <= 5).sum()
    in_top_10 = (df['ground_truth_transformer_rank'] <= 10).sum()
    in_top_20 = (df['ground_truth_transformer_rank'] <= 20).sum()
    in_top_50 = (df['ground_truth_transformer_rank'] <= 50).sum()
    in_top_100 = (df['ground_truth_transformer_rank'] <= 100).sum()
    in_top_150 = (df['ground_truth_transformer_rank'] <= 150).sum()
    in_top_200 = (df['ground_truth_transformer_rank'] <= 200).sum()

    print(f"  In Top 5: {in_top_5} ({in_top_5/total_entries*100:.2f}%)")
    print(f"  In Top 10: {in_top_10} ({in_top_10/total_entries*100:.2f}%)")
    print(f"  In Top 20: {in_top_20} ({in_top_20/total_entries*100:.2f}%)")
    print(f"  In Top 50: {in_top_50} ({in_top_50/total_entries*100:.2f}%)")
    print(f"  In Top 100: {in_top_100} ({in_top_100/total_entries*100:.2f}%)")
    print(f"  In Top 150: {in_top_150} ({in_top_150/total_entries*100:.2f}%)")
    print(f"  In Top 200: {in_top_200} ({in_top_200/total_entries*100:.2f}%)")
else:
    print(f"  Not Found (N/A): {num_na} ({na_percentage:.2f}%)")




TRANSFORMER SCORE N/A COUNT: 413 out of 1724 (23.96%)

Ground Truth Rank Distribution:
  In Top 5: 753 (43.68%)
  In Top 10: 895 (51.91%)
  In Top 20: 1007 (58.41%)
  In Top 50: 1130 (65.55%)
  In Top 100: 1226 (71.11%)
  In Top 150: 1280 (74.25%)
  In Top 200: 1311 (76.04%)


### 3.3 Error Analysis

Our final method produces 455 errors, which can be categorized into two distinct sources:

#### Source 1: Transformer Candidate Limitation (413 errors)

The majority of errors occur when the correct word is not present in the top-200 transformer predictions. This represents a fundamental limitation of the language model's ability to generate appropriate candidates given the sentential context.

**Potential Solutions:**
1. **Advanced Models**: Upgrade to more powerful language models (e.g., BERT-large, RoBERTa, GPT-based models) that may better capture context and generate more accurate candidate sets
2. **Enhanced Context**: Provide broader context beyond single sentences, such as surrounding sentences or document-level information
3. **Hybrid Generation**: Generate all linguistically valid words within n edit distance of the typo and use the transformer only for scoring/ranking, rather than candidate generation

#### Source 2: Selection Method Errors (42 errors)

A smaller but non-trivial portion of errors arise from suboptimal weighting of our three scoring components (edit distance, similarity, transformer score), even when the correct word is available.

**Case Study 1: Edit Distance Dominance**
- Typo: "crrntrues"
- Ground truth: "centuries" (transformer: 0.340, edit distance: 0.43)
- Selected: "countries" (transformer: 0.00002, edit distance: **0.57**)
- **Issue**: Edit distance score (0.57 vs 0.43) overrode the significantly stronger transformer score (0.340 vs 0.00002)

**Case Study 2: Similarity Score Dominance**  
- Typo: "CChtios"
- Ground truth: "Catholics" (similarity: moderate)
- Selected: "Doctors" (similarity: **high**)
- **Issue**: High character-level similarity outweighed semantic/contextual appropriateness

**Analysis:**

These errors reveal a fundamental tension in our weighting scheme. While increasing the transformer weight (currently 0.1) might resolve some cases, our experiments show this often degrades overall performance because:
1. Transformer scores are not consistently reliable indicators across all contexts
2. Edit distance remains the strongest general signal for spelling correction
3. Many correct words have relatively low transformer scores despite being contextually appropriate

**Proposed Solution:**

Rather than manual weight tuning, a more robust approach would be to learn optimal weights from data:
- Train a supervised model (e.g., logistic regression, SVM, or small neural network)
- Features: edit distance score, similarity score, transformer score, and potentially their interactions
- Training data: pairs of (typo, candidates) with labeled correct choices
- This would learn context-dependent weighting strategies that adapt to different error patterns

In [3]:
import pandas as pd
import numpy as np

# Read the scores.csv file
df = pd.read_csv('../output/dev_score_details.csv')

print("\n" + "="*80)
print("INCORRECT PREDICTIONS")
print("="*80)

incorrect_df = df[df['selected'] != df['ground_truth']]
num_incorrect = len(incorrect_df)
incorrect_percentage = (num_incorrect / total_entries) * 100

print(f"\nTotal Incorrect: {num_incorrect} out of {total_entries} ({incorrect_percentage:.2f}%)")
print(f"Total Correct: {total_entries - num_incorrect} ({100 - incorrect_percentage:.2f}%)")


INCORRECT PREDICTIONS

Total Incorrect: 455 out of 1724 (26.39%)
Total Correct: 1269 (73.61%)


In [4]:
import pandas as pd
import numpy as np

# Read the scores.csv file
df = pd.read_csv('../output/dev_score_details.csv')

print("\n\n" + "="*80)
print("INCORRECT PREDICTIONS (Ground Truth WAS in Top-200)")
print("="*80)

incorrect_with_score_df = df[
    (df['selected'] != df['ground_truth']) & 
    (df['ground_truth_transformer_score'].notna())
]
num_incorrect_with_score = len(incorrect_with_score_df)
incorrect_with_score_percentage = (num_incorrect_with_score / total_entries) * 100

print(f"\nIncorrect (ground truth available): {num_incorrect_with_score} out of {total_entries} ({incorrect_with_score_percentage:.2f}%)")
print(f"Incorrect (ground truth NOT available): {num_incorrect - num_incorrect_with_score}")

if num_incorrect_with_score > 0:
    print("\n" + incorrect_with_score_df[['typo', 'ground_truth', 'ground_truth_transformer_score',
                                          'ground_truth_edit_distance', 'ground_truth_edit_score','ground_truth_similarity_score',
                                          'selected', 'selected_transformer_score',
                                          'selected_edit_distance', 'selected_edit_score', 'selected_similarity_score']].to_string())




INCORRECT PREDICTIONS (Ground Truth WAS in Top-200)

Incorrect (ground truth available): 42 out of 1724 (2.44%)
Incorrect (ground truth NOT available): 413

            typo ground_truth  ground_truth_transformer_score  ground_truth_edit_distance  ground_truth_edit_score  ground_truth_similarity_score    selected  selected_transformer_score  selected_edit_distance  selected_edit_score  selected_similarity_score
40       CChtios    Catholics                        0.332139                           5                 0.285714                       0.375000     Doctors                    0.000346                       5             0.285714                   0.571429
62        wantsd       wanted                        0.090196                           1                 0.857143                       0.833333       wants                    0.000585                       1             0.857143                   0.909091
63         Theee        These                        0.068274      